<a href="https://colab.research.google.com/github/lynnkaram/health-informatics-projects/blob/main/genomics-final-project/genomics_sequence_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project 2: De Bruijn Graph Construction from Sequencing Reads
*Lynn Karam, student ID: LK732*

This script constructs a De Bruijn graph from a set of DNA sequencing reads. The graph represents the connectivity of k-mers (specifically 5-mers) derived from the input data.

Key Concepts:
- The graph nodes will represent all unique 4-mers (substrings of length k-1).
- The directed edges will correspond to all unique 5-mers (substrings of length k). These are directed connections representing 5-mers, going from prefix to suffix.
- Multiplicity: Count of how many times each 5-mer appears

In [ ]:
from collections import defaultdict
from typing import List, Dict, Tuple

def extract_kmers(reads: List[str], k: int = 5) -> List[str]:
    """
    Extract all k-mers (substrings of length k) from the given sequencing reads.

    Definitions:
      - A "k-mer" is any contiguous substring of length k
      - If a read has length L, it contains (L - k + 1) k-mers

    Args:
        reads: List of DNA sequencing reads (strings over {A,C,G,T})
        k: Length of k-mers (for this project, k = 5).

    Returns:
        A list containing every k-mer extracted from all reads (including repeats)
        Repeats matter because they determine edge multiplicity later

    Example:
        read = "TTGCAT", k=5 -> ["TTGCA", "TGCAT"]
    """
    kmers = []

    # Loop over reads one-by-one
    for read in reads:
        # Slide a window of size k across the read
        # i runs over all start positions where a length-k substring fits
        for i in range(len(read) - k + 1):
            kmers.append(read[i:i + k])

    return kmers


def build_debruijn_graph(kmers: List[str]) -> Tuple[Dict[str, str], Dict[Tuple[str, str], Dict]]:
    """
    Build a De Bruijn graph from a list of k-mers

    De Bruijn Graph (for fixed k):
      - Nodes are unique (k-1)-mers
      - Each k-mer defines a directed edge:
            prefix = kmer[0 : k-1]
            suffix = kmer[1 : k]
        Edge goes prefix -> suffix
      - Edge multiplicity = number of times that k-mer appears across all reads

    Representation used here:
      - nodes: dict mapping node sequence -> node sequence (acts like an ordered set)
      - edges: dict mapping (prefix, suffix) -> {"sequence": kmer, "count": multiplicity}

    Why (prefix, suffix) is enough as an edge key:
      - For DNA strings, a given (k-1)-prefix and (k-1)-suffix uniquely determine the k-mer,
        because the overlap is k-2 characters. So we can safely aggregate counts under that key

    Args:
        kmers: List of k-mers (all same length k).

    Returns:
        nodes: dictionary of unique (k-1)-mer node sequences
        edges: dictionary of directed edges with stored k-mer and multiplicity
    """
    # Using a dict for nodes to preserve uniqueness easily; keys are the 4-mers
    nodes: Dict[str, str] = {}

    # edges[(prefix, suffix)] stores:
    #   - sequence: the representative k-mer for that edge
    #   - count: how many times it occurred (multiplicity)
    edges = defaultdict(lambda: {"sequence": None, "count": 0})

    for kmer in kmers:
        # For k=5:
        # prefix/suffix are length 4 (k-1) and become graph nodes
        prefix = kmer[:-1]   # first 4 chars
        suffix = kmer[1:]    # last  4 chars

        # Add nodes (unique (k-1)-mers)
        nodes[prefix] = prefix
        nodes[suffix] = suffix

        # Add / update directed edge (prefix -> suffix) and count multiplicity
        edge_key = (prefix, suffix)
        edges[edge_key]["sequence"] = kmer      # this edge corresponds to this k-mer
        edges[edge_key]["count"] += 1           # multiplicity

    return nodes, edges


def format_output(nodes: Dict[str, str], edges: Dict[Tuple[str, str], Dict]) -> str:
    """
    Format the graph output to match the assignment example

    Output format:
      NODES (4-mers):
      N1: <node_seq>
      ...

      EDGES (5-mers):
      E1: N# -> N# | Sequence: <kmer> | Count: <multiplicity>
      ...

    Notes:
      - Nodes are sorted alphabetically for consistent IDs across runs
      - Edges are sorted by (source_seq, target_seq) for consistent ordering
    """
    output = []

    # Nodes section
    output.append("NODES (4-mers):")

    # Sort node sequences so node IDs (N1, N2, ...) are stable and reproducible
    sorted_nodes = sorted(nodes.keys())

    # Map node sequence -> assigned node ID (e.g., "AACT" -> "N1")
    node_to_id: Dict[str, str] = {}
    for idx, node_seq in enumerate(sorted_nodes, start=1):
        node_id = f"N{idx}"
        node_to_id[node_seq] = node_id
        output.append(f"{node_id}: {node_seq}")

    output.append("")  # blank line between sections

    # Edges section
    output.append("EDGES (5-mers):")

    # Sort edges for consistent output
    # edges.items() gives: ((src_seq, tgt_seq), data_dict)
    sorted_edges = sorted(edges.items(), key=lambda x: (x[0][0], x[0][1]))

    for idx, ((src_seq, tgt_seq), data) in enumerate(sorted_edges, start=1):
        src_id = node_to_id[src_seq]
        tgt_id = node_to_id[tgt_seq]
        output.append(
            f"E{idx}: {src_id} -> {tgt_id} | "
            f"Sequence: {data['sequence']} | Count: {data['count']}"
        )

    return "\n".join(output)

def main():
    """
    Driver function:
      1) Loads reads
      2) Extracts all k-mers (k=5)
      3) Builds de Bruijn graph (nodes are 4-mers, edges are 5-mers)
      4) Prints summary statistics + formatted graph
    """
    reads = [
        "TTGCAT",
        "TGAACT",
        "ATTGCA",
        "TGCATT",
        "CATTTG",
        "TGGTAA",
        "ATTTGA",
        "AACTAG",
        "TAGCAT",
        "TAGGTA",
        "GCATTT",
        "TTGAAC",
        "TAGGTA",
        "GGTAGC",
        "GTAGCA",
        "AGCATT"
    ]

    k = 5  # assignment specifies k=5

    # Step 1: Extract all 5-mers (including repeats)
    kmers = extract_kmers(reads, k)

    # Step 2: Build the de Bruijn graph using those 5-mers
    nodes, edges = build_debruijn_graph(kmers)

    # Summary stats
    # Total k-mers: total windows extracted across all reads
    total_kmers = len(kmers)

    # Unique k-mers: how many distinct 5-mers exist (edges after collapsing)
    unique_kmers = len(set(kmers))

    # Nodes: number of distinct 4-mers encountered as prefixes/suffixes
    num_nodes = len(nodes)

    # Edges: number of distinct directed (prefix->suffix) connections (unique 5-mers)
    num_edges = len(edges)

    print(f"Total 5-mers extracted from reads: {total_kmers}")
    print(f"Unique 5-mers: {unique_kmers}")
    print(f"De Bruijn graph number of nodes (unique 4-mers): {num_nodes}")
    print(f"De Bruijn graph number of edges (unique 5-mer connections): {num_edges}")
    print()
    print(format_output(nodes, edges))

if __name__ == "__main__":
    main()

Total 5-mers extracted from reads: 32
Unique 5-mers: 20
De Bruijn graph number of nodes (unique 4-mers): 21
De Bruijn graph number of edges (unique 5-mer connections): 20

NODES (4-mers):
N1: AACT
N2: ACTA
N3: AGCA
N4: AGGT
N5: ATTG
N6: ATTT
N7: CATT
N8: CTAG
N9: GAAC
N10: GCAT
N11: GGTA
N12: GTAA
N13: GTAG
N14: TAGC
N15: TAGG
N16: TGAA
N17: TGCA
N18: TGGT
N19: TTGA
N20: TTGC
N21: TTTG

EDGES (5-mers):
E1: N1 -> N2 | Sequence: AACTA | Count: 1
E2: N2 -> N8 | Sequence: ACTAG | Count: 1
E3: N3 -> N10 | Sequence: AGCAT | Count: 2
E4: N4 -> N11 | Sequence: AGGTA | Count: 2
E5: N5 -> N20 | Sequence: ATTGC | Count: 1
E6: N6 -> N21 | Sequence: ATTTG | Count: 2
E7: N7 -> N6 | Sequence: CATTT | Count: 2
E8: N9 -> N1 | Sequence: GAACT | Count: 1
E9: N10 -> N7 | Sequence: GCATT | Count: 3
E10: N11 -> N12 | Sequence: GGTAA | Count: 1
E11: N11 -> N13 | Sequence: GGTAG | Count: 1
E12: N13 -> N14 | Sequence: GTAGC | Count: 2
E13: N14 -> N3 | Sequence: TAGCA | Count: 2
E14: N15 -> N4 | Sequence: TAGGT

## Results and Observations

The De Bruijn graph was successfully constructed from 16 sequencing reads with k = 5.
From these reads:
- 32 total 5-mers were extracted (including repeats),
- 20 unique 5-mers formed 20 directed edges,
- 21 unique 4-mer nodes were identified.

Notable observations:
- Edge E9 (GCATT) has the highest multiplicity (count = 3), indicating that this 5-mer appears in multiple reads.
- Several other edges have multiplicity 2, reflecting repeated prefix-suffix overlaps across the dataset.
- The resulting graph captures the overlap structure of the reads and, in principle, could be used for sequence reconstruction by identifying valid paths through the nodes that respect edge multiplicities.
- This De Bruijn graph representation efficiently handles repeated sequences and provides a foundation for genome assembly algorithms.
